# Data, Annotation, and Ground Truth — companion notebook

Chapter 5 of *Applied Natural Language Processing — From Tokens to Agents*.
Read the chapter first: **[Data, Annotation, and Ground Truth](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html)**.

Chapter 4 compared representations against a column called `sentiment`. This chapter asks
where that column came from and what the scores mean if part of it is wrong. It keeps
chapter 4's split and its highest-scoring model fixed, and changes the labels. This
notebook runs those experiments in the chapter's order:

1. **The frozen model**: chapter 4's split and its word unigram + bigram TF-IDF row.
2. **Read a hundred rows**: the random sample the chapter read by hand, and the arithmetic
   of the rule of three.
3. **The schema**: a four-word screening rule for reviews that may hold two verdicts.
4. **Agreement**: raw agreement, Cohen's kappa and three other coefficients from one 2×2
   table, and what a class-balanced subsample does to them.
5. **Auditing labels at scale**: out-of-fold flags, what dropping or flipping the flagged
   rows does, and the same operation on random rows as a control.
6. **Label noise in training**, by direction and by count.
7. **Label noise in the test set**, and what a small gap between two models means.
8. **Leakage**: exact and near duplicates, and controlled contamination.
9. **The budget**: fewer clean labels against more noisy ones, and uncertainty sampling.

**How to read the numbers.** Everything printed below is what *this run* produced. Where
the chapter prints the same quantity, the cell prints the book's value beside it. Three
kinds of comparison appear, and each cell says which one applies:

- *deterministic* quantities (counts, the frozen model, fixed-seed experiments whose code
  the chapter prints or describes exactly) are compared at the book's printed precision;
- *experiments with random draws whose seeds the chapter does not print* (the noise
  tables, the budget grid, uncertainty sampling) are run with this notebook's own seeds,
  and the cell reports the distance from the book's value next to the seed-to-seed spread;
- *human judgements* (which of 100 rows is an error, which of 40 disputes is a schema
  problem) cannot be reproduced by code. The notebook shows you the same rows and gives
  the chapter's counts as the chapter's.

**One thing this notebook cannot reproduce.** The chapter's second "annotator" is a lexicon
rule with 36 positive and 37 negative phrases, and the chapter does not print the lists.
Its 2×2 table is therefore taken as given, and every coefficient derived from it is
recomputed. To keep that section runnable end to end, the notebook also defines a small
rule of its own (22 phrases, the one from chapter 1's notebook) and repeats the analysis
on it. That rule's numbers are not the chapter's.

**Runtime and downloads.** About 2 minutes on a laptop CPU (Apple M-series). The only
download is the dataset (24 MB), cached under `data/cache/` next to this notebook. The
model is a linear classifier over sparse features, so even the experiments that retrain
it more than a hundred times take seconds. No cell is heavy, and the `ANLP_SKIP_HEAVY`
switch of the other notebooks changes nothing here.

## 1. Setup

The first cell records the interpreter and package versions this run used. If your
numbers differ from the ones saved in this notebook, compare this cell first.

In [1]:
import hashlib
import platform
import re
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp
from scipy.stats import binom
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, cohen_kappa_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "pandas", "scipy", "scikit-learn", "pyarrow", "requests"]:
    print(f"{pkg:<13} {version(pkg)}")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
pandas        3.0.5
scipy         1.18.1
scikit-learn  1.9.0
pyarrow       25.0.1
requests      2.34.2


Three helpers used throughout.

- `vs_book` prints this run's number beside the book's and compares them at the book's
  printed precision.
- `vs_book_near` is for experiments run with this notebook's own random draws. It prints
  the difference from the book's value; where a spread over seeds is known, read the
  difference against it.
- `scores` returns accuracy, macro-F1 and the F1 of the negative class, in percent.

In [2]:
def vs_book(label, ours, book, fmt="{}", note=None):
    """This run's value beside the book's, compared at the book's printed precision."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    if note:
        verdict = f"not comparable: {note}"
    else:
        verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<40} this run {ours_s:>10}   book {book_s:>10}   {verdict}")


def vs_book_near(label, ours, book, fmt="{:.2f}", note=None):
    """For experiments with this notebook's own random draws: the distance from the book's value."""
    verdict = f"not comparable: {note}" if note else f"{ours - book:+.2f} from the book"
    print(f"{label:<40} this run {fmt.format(ours):>10}   book {fmt.format(book):>10}   {verdict}")


def scores(y_true, y_pred):
    """Accuracy, macro-F1 and negative-class F1, in percent."""
    return (100 * accuracy_score(y_true, y_pred),
            100 * f1_score(y_true, y_pred, average="macro", zero_division=0),
            100 * f1_score(y_true, y_pred, pos_label=0, zero_division=0))


def macro_f1(y_true, y_pred):
    return 100 * f1_score(y_true, y_pred, average="macro", zero_division=0)

## 2. The number you have been trusting

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#the-number-you-have-been-trusting)

The data, the split and the model are chapter 4's, unchanged: the Turkish product reviews
of `fthbrmnby/turkish_product_reviews` (CC BY-SA 4.0), a stratified sample of 50,000 after
de-duplication, split 40,000 / 10,000 with seed 42, and TF-IDF over word unigrams and
bigrams with a class-weighted logistic regression. [Chapter 4's notebook](https://github.com/bbardakk/uygulamali-dogal-dil-isleme-notebooks/blob/main/en/chapters/04-bake-off/04-bake-off.ipynb) builds each of these
step by step; here they are one cell.

The chapter calls the labels in the `sentiment` column the **reference labels**: the
point of reference for evaluation, not a claim to be the one indisputable reading of the
text.

**What to look at:** the fingerprints say you have the same rows as chapter 4, and the
three scores are the row every experiment below starts from.

In [3]:
URL = ("https://huggingface.co/datasets/fthbrmnby/turkish_product_reviews/"
       "resolve/main/data/train-00000-of-00001.parquet")
CACHE = Path("data/cache/turkish_product_reviews.parquet")

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(URL, timeout=300)
    response.raise_for_status()
    CACHE.write_bytes(response.content)

# chapter 4's loading code and split
raw = pd.read_parquet(CACHE).rename(columns={"sentence": "text", "sentiment": "label"})
raw["text"] = raw["text"].astype(str).str.strip()
df = raw.drop_duplicates(subset="text")
sub, _ = train_test_split(df, train_size=50_000, stratify=df["label"], random_state=42)
train, test = train_test_split(sub, test_size=0.2, stratify=sub["label"], random_state=42)
y_train, y_test = train["label"].to_numpy(), test["label"].to_numpy()

def fingerprint(frame):
    rows = ",".join(map(str, sorted(frame.index)))
    return hashlib.sha256(rows.encode()).hexdigest()[:12]

# chapter 4's tokenisation and its highest-scoring row
TR_LOWER = str.maketrans({"I": "ı", "İ": "i"})
def tr_lower(s):
    return s.translate(TR_LOWER).lower()

PAT = r"[a-zçğıöşü]+"
LOC = dict(preprocessor=tr_lower, lowercase=False)
TOKEN_RE = re.compile(PAT)
def tokenize(s):
    return TOKEN_RE.findall(tr_lower(s))

def make_vec():
    return TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2), token_pattern=PAT, **LOC)

def clf(C=1.0):
    return LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=42)

vec = make_vec()
Xtr = vec.fit_transform(train["text"])
Xte = vec.transform(test["text"])
model = clf().fit(Xtr, y_train)
pred = model.predict(Xte)
prob = model.predict_proba(Xte)[:, 1]              # the model's score for "positive"

vs_book("train fingerprint (chapter 4)", fingerprint(train), "8205c2be5ebf")
vs_book("test fingerprint (chapter 4)", fingerprint(test), "634cdb3bb002")
vs_book("features", Xtr.shape[1], 101_659, "{:,}")
base_acc, base_macro, base_f1n = scores(y_test, pred)
vs_book("accuracy", base_acc, 92.57, "{:.2f}")
vs_book("macro-F1", base_macro, 76.21, "{:.2f}")
vs_book("F1 (negative)", base_f1n, 56.47, "{:.2f}")

train fingerprint (chapter 4)            this run 8205c2be5ebf   book 8205c2be5ebf   same as the book
test fingerprint (chapter 4)             this run 634cdb3bb002   book 634cdb3bb002   same as the book
features                                 this run    101,659   book    101,659   same as the book
accuracy                                 this run      92.57   book      92.57   same as the book
macro-F1                                 this run      76.21   book      76.21   same as the book
F1 (negative)                            this run      56.47   book      56.47   same as the book


## 3. Read a hundred rows. Yourself.

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#sec-random-audit)

Before any tool, the chapter draws 100 test rows at random and reads them. The draw is
`np.random.default_rng(42).choice(10_000, 100, replace=False)` over the positions of the
test set, so you can read the same hundred.

What the reader decided about each row is a human judgement, and code cannot reproduce
it. The chapter's counts, by one reader: **0** outright errors (the label contradicts the
text), **7** undecidable under a binary product-verdict schema (3 of them "not yet
received"), **9** arguable either way, **84** ok. The cell checks what it can: the draw
contains the row the chapter quotes, and 98 of the 100 are labelled positive.

**What to look at:** read the rows before you read the chapter's counts, and write down
your own criteria first. The chapter's third exercise asks for exactly that.

In [4]:
audit_rows = np.random.default_rng(42).choice(len(test), 100, replace=False)
vs_book("positive reference labels in the sample", int(y_test[audit_rows].sum()), 98)
vs_book("the draw contains test row 4745", 4745 in set(audit_rows.tolist()), True)
print(f"\nrow 4745, label {y_test[4745]}: {test['text'].iloc[4745]!r}\n")

print("the hundred rows   (position in the test set, reference label, text)")
for position in audit_rows:
    print(f"{position:>5}  {y_test[position]}  {test['text'].iloc[position][:150]}")

positive reference labels in the sample  this run         98   book         98   same as the book
the draw contains test row 4745          this run       True   book       True   same as the book

row 4745, label 1: 'sipariş verdim umarım tez ulaşır :)'

the hundred rows   (position in the test set, reference label, text)
 4363  1  öncelikle, ürünün kalitesine söyleyecek hiç birşey yok, gerçekten kaliteli. i̇stediğiniz tüm kahveleri yapması da ayrıcalıklı tarafı. filtre kahve kıs
 9385  1  ürünü salıgünü sipariş verdim cuma günü elimdeydi sorunsuz teslimattan dolayı hb.ye tşk.kokuya gelince güzel koku ama kalıcılıgı daha ön planda kokuyu
 6684  1  1 gün de geldi hızlı teslimat için teşekkürler
 4645  1  kırtasiye marketlerde 2 katından fazlasına satılmakta ..bana beyaz renk olan gönderildi..istediğim de oydu.. enfes.. uzun yıllar kullanılabilecek bir 
 4473  1  süper kalite güzel görünüm. memnun edici.
 7744  1  çocuğuma akşamları okuyorum. beraber hem güzel vakit geçiriyoruz hemde eği

Zero outright errors in 100 rows does not mean an error rate of zero. The chapter's
arithmetic:

- the **rule of three**: with zero events in $n$ independent draws, $3/n$ is an
  approximate one-sided 95% upper bound on the rate;
- with a uniform prior, the posterior mean after zero events in 100 is $1/(n+2)$;
- 98 positives in 100 against a corpus rate of 93.7%: the chance of at most 2 negatives
  in 100 draws at a 6.3% negative rate.

The rule of three applies only when the count is zero. If you find one or more errors in
your own sample, compute a binomial interval instead (the chapter uses Wilson's).

In [5]:
n = 100
exact_bound = 1 - 0.05 ** (1 / n)                  # solves (1 - p)^n = 0.05
vs_book("rule of three, 3/n, %", 100 * 3 / n, 3.0, "{:.1f}")
print(f"  the exact bound it approximates: {100 * exact_bound:.2f}%")
vs_book("posterior mean 1/(n+2), %", 100 / (n + 2), 0.98, "{:.2f}")
vs_book("P(at most 2 negatives in 100), %", 100 * binom.cdf(2, n, 0.063), 4.5, "{:.1f}")


def wilson(k, n, z=1.96):
    """Wilson score interval for a proportion k/n."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


print("\nif the count is not zero, e.g. 2 errors in 100:  Wilson 95% interval "
      + "{:.1f}% to {:.1f}%".format(*(100 * x for x in wilson(2, 100))))

rule of three, 3/n, %                    this run        3.0   book        3.0   same as the book
  the exact bound it approximates: 2.95%
posterior mean 1/(n+2), %                this run       0.98   book       0.98   same as the book
P(at most 2 negatives in 100), %         this run        4.5   book        4.5   same as the book

if the count is not zero, e.g. 2 errors in 100:  Wilson 95% interval 0.6% to 7.0%


## 4. The schema can do as much damage as the annotators

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#the-schema-can-do-as-much-damage-as-the-annotators)

A review can praise the product and complain about the delivery. A single
positive/negative bit has to drop one of the two. To find *candidates* for such reviews,
the chapter marks four Turkish words that all mean roughly "but". The rule is a screening
device: whether a matching review really holds two opposed verdicts is not known without
reading it, and the rule's precision is not measured.

**What to look at:** how large the matching group is, its share of negative labels, and
the model's scores on the two groups.

In [6]:
# tr_lower is the Turkish-aware lowercaser from chapter 4 (İ->i, I->ı)
CONTRAST = re.compile(r"\b(ama|fakat|ancak|lakin)\b")
has_but = np.array([bool(CONTRAST.search(tr_lower(t))) for t in test.text])

vs_book("reviews matching the rule", int(has_but.sum()), 2_003, "{:,}")
BOOK_CONTRAST = {True: (20.03, 11.48, 84.07, 72.11, 53.84), False: (79.97, 5.03, 94.70, 77.72, 58.27)}
for group, name in [(True, "rule matches"), (False, "no match")]:
    mask = has_but == group
    acc, macro, f1n = scores(y_test[mask], pred[mask])
    book = BOOK_CONTRAST[group]
    print(f"\n{name}")
    vs_book("  share of test set, %", 100 * mask.mean(), book[0], "{:.2f}")
    vs_book("  reference negative rate, %", 100 * (y_test[mask] == 0).mean(), book[1], "{:.2f}")
    vs_book("  model accuracy", acc, book[2], "{:.2f}")
    vs_book("  model macro-F1", macro, book[3], "{:.2f}")
    vs_book("  model F1 (negative)", f1n, book[4], "{:.2f}")

reviews matching the rule                this run      2,003   book      2,003   same as the book

rule matches
  share of test set, %                   this run      20.03   book      20.03   same as the book
  reference negative rate, %             this run      11.48   book      11.48   same as the book
  model accuracy                         this run      84.07   book      84.07   same as the book
  model macro-F1                         this run      72.11   book      72.11   same as the book
  model F1 (negative)                    this run      53.84   book      53.84   same as the book

no match
  share of test set, %                   this run      79.97   book      79.97   same as the book
  reference negative rate, %             this run       5.03   book       5.03   same as the book
  model accuracy                         this run      94.70   book      94.70   same as the book
  model macro-F1                         this run      77.72   book      77.72   same as the b

**Reading it.** A fifth of the test set matches, it has more than twice the negative
rate, and the model scores lower on it. The chapter is careful about what that does and
does not show, and measures three things before interpreting it:

- how uncertain the macro-F1 gap is (a bootstrap over the test rows);
- whether the two groups differ in something else (they do: length);
- how much of the rule rests on *ancak*, which also means "only" or "barely".

In [7]:
rng = np.random.default_rng(42)
gaps = []
for _ in range(5_000):
    i = rng.integers(0, len(y_test), len(y_test))
    m = has_but[i]
    gaps.append(macro_f1(y_test[i][~m], pred[i][~m]) - macro_f1(y_test[i][m], pred[i][m]))
observed = macro_f1(y_test[~has_but], pred[~has_but]) - macro_f1(y_test[has_but], pred[has_but])
vs_book("macro-F1 gap, no match − match", observed, 5.6, "{:.1f}")
vs_book("  bootstrap 95%, lower end", np.percentile(gaps, 2.5), 2.5, "{:.1f}")
vs_book("  bootstrap 95%, upper end", np.percentile(gaps, 97.5), 8.8, "{:.1f}")

n_tokens = np.array([len(tokenize(t)) for t in test.text])
vs_book("median tokens, rule matches", int(np.median(n_tokens[has_but])), 30)
vs_book("median tokens, no match", int(np.median(n_tokens[~has_but])), 16)

lowered = [tr_lower(t) for t in test.text]
with_ancak = np.array([bool(re.search(r"\bancak\b", t)) for t in lowered])
others = np.array([bool(re.search(r"\b(ama|fakat|lakin)\b", t)) for t in lowered])
vs_book("reviews containing 'ancak'", int(with_ancak.sum()), 351)
vs_book("  matching through 'ancak' alone", int((with_ancak & ~others).sum()), 287)
print(f"macro-F1 with 'ancak' taken out of the rule: match {macro_f1(y_test[others], pred[others]):.2f}, "
      f"no match {macro_f1(y_test[~others], pred[~others]):.2f}")

macro-F1 gap, no match − match           this run        5.6   book        5.6   same as the book
  bootstrap 95%, lower end               this run        2.5   book        2.5   same as the book
  bootstrap 95%, upper end               this run        8.8   book        8.8   same as the book
median tokens, rule matches              this run         30   book         30   same as the book
median tokens, no match                  this run         16   book         16   same as the book
reviews containing 'ancak'               this run        351   book        351   same as the book
  matching through 'ancak' alone         this run        287   book        287   same as the book
macro-F1 with 'ancak' taken out of the rule: match 71.29, no match 77.80


**Reading it.** The gap is there, with a wide interval, and it survives taking *ancak*
out of the rule. But the matching reviews are also about twice as long, and no review
here was re-annotated aspect by aspect. So the measurement does not choose between "the
schema compresses two verdicts into one bit" and "the model reads these sentences badly".
The chapter's answer to the schema side is a written guideline with decided cases; that
part of the chapter is prose and has no code.

## 5. Agreement: what does 93% tell you by itself?

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#agreement-and-why-93-of-it-can-be-worthless)

To know whether labels can be trusted you need a second opinion on the same items and a
number for how often the two match. The chapter had no second human annotator, so it
compares the reference labels (A) with a lexicon rule (B) on the 10,000 test rows. It
says plainly what that is: an overlap between the reference labels and a rule, written by
someone who had seen the labels. It demonstrates the arithmetic; it is not a study of
human agreement.

The rule's phrase lists are not printed in the chapter, so its table is taken as given
here. Everything derived from the table is recomputed.

**What to look at:** raw agreement is 93.42%, and the agreement expected if A and B
labelled independently at their own rates is already 90.99%.

In [8]:
# the chapter's table: A (reference, rows) vs B (lexicon rule, columns), negative first
BOOK_TABLE = np.array([[141, 491],
                       [167, 9201]], dtype=float)


def coefficients(table):
    """Agreement coefficients for two raters and two categories, from a 2x2 table."""
    table = np.asarray(table, dtype=float)
    n = table.sum()
    p_o = np.trace(table) / n
    a_rates, b_rates = table.sum(axis=1) / n, table.sum(axis=0) / n
    p_e = float(a_rates @ b_rates)                              # each rater keeps their own rate
    kappa = (p_o - p_e) / (1 - p_e)
    # Krippendorff's alpha, nominal: disagreement observed / expected from the pooled labels
    pooled = table + table.T
    totals, N = pooled.sum(axis=1), pooled.sum()
    alpha = 1 - ((N - np.trace(pooled)) / N) / ((N * N - (totals ** 2).sum()) / (N * (N - 1)))
    mean_rates = (a_rates + b_rates) / 2
    chance_ac1 = 2 * mean_rates[0] * mean_rates[1]              # Gwet's AC1, two categories
    return {"p_o": p_o, "p_e": p_e, "kappa": kappa, "alpha": alpha,
            "AC1": (p_o - chance_ac1) / (1 - chance_ac1), "PABAK": 2 * p_o - 1,
            "A positive": a_rates[1], "B positive": b_rates[1]}


book = coefficients(BOOK_TABLE)
vs_book("A calls positive, %", 100 * book["A positive"], 93.68, "{:.2f}")
vs_book("B calls positive, %", 100 * book["B positive"], 96.92, "{:.2f}")
vs_book("raw agreement p_o", book["p_o"], 0.9342, "{:.4f}")
vs_book("expected agreement p_e", book["p_e"], 0.9099, "{:.4f}")
print("\n### the identical 10,000-row table, five coefficients")
vs_book("raw agreement  p_o", book["p_o"], 0.9342, "{:.4f}")
vs_book("Gwet's AC1", book["AC1"], 0.9277, "{:.4f}")
vs_book("PABAK  (2*p_o - 1)", book["PABAK"], 0.8684, "{:.4f}")
vs_book("Cohen's kappa", book["kappa"], 0.2698, "{:.4f}")
vs_book("Krippendorff's alpha", book["alpha"], 0.2655, "{:.4f}")

A calls positive, %                      this run      93.68   book      93.68   same as the book
B calls positive, %                      this run      96.92   book      96.92   same as the book
raw agreement p_o                        this run     0.9342   book     0.9342   same as the book
expected agreement p_e                   this run     0.9099   book     0.9099   same as the book

### the identical 10,000-row table, five coefficients
raw agreement  p_o                       this run     0.9342   book     0.9342   same as the book
Gwet's AC1                               this run     0.9277   book     0.9277   same as the book
PABAK  (2*p_o - 1)                       this run     0.8684   book     0.8684   same as the book
Cohen's kappa                            this run     0.2698   book     0.2698   same as the book
Krippendorff's alpha                     this run     0.2655   book     0.2655   same as the book


**Reading it.** Five coefficients, one table, values from 0.27 to 0.93. They answer
different questions about chance: kappa lets each rater keep their own rate, alpha pools
the two raters' labels, AC1 and PABAK assume much less agreement by chance. Reporting one
of them without the table and the two raters' rates hides which question was asked.

Kappa near alpha on this table is not an independent confirmation. Both are computed from
the same label pairs.

### A rule of our own, so the rest can be run

The chapter's next step needs B's label for every row, which the table does not contain.
So here is a small transparent rule: 12 negative and 10 positive phrases, the one chapter
1's notebook uses. It has the same shape as the chapter's (negative only if a negative
phrase matches and no positive one does). **Its numbers are not the chapter's.**

In [9]:
NEG = ["kötü", "berbat", "rezalet", "değil", "beğenmedim", "memnun kalmadım",
       "tavsiye etmem", "tavsiye etmiyorum", "iade", "bozuk", "kalitesiz", "pişman"]
POS = ["güzel", "harika", "mükemmel", "süper", "memnunum", "teşekkür",
       "tavsiye ederim", "beğendim", "kaliteli", "başarılı"]


def annotator_b(text):
    t = tr_lower(text).replace("̇", "")       # chapter 4: the stray combining dot
    neg = any(p in t for p in NEG)
    pos = any(p in t for p in POS)
    if neg and not pos:
        return 0
    return 1          # no match, or matches of both kinds: positive


def table_of(a, b):
    return np.array([[((a == i) & (b == j)).sum() for j in (0, 1)] for i in (0, 1)], dtype=float)


b_ours = np.array([annotator_b(t) for t in test["text"]])
ours_table = table_of(y_test, b_ours)
print("A (reference, rows) vs our 22-phrase rule (columns), n = 10,000\n")
print("                B: negative   B: positive")
for i, name in enumerate(["A: negative", "A: positive"]):
    print(f"{name}     {int(ours_table[i, 0]):>9,}     {int(ours_table[i, 1]):>9,}")
ours = coefficients(ours_table)
print(f"\nraw agreement {100 * ours['p_o']:.2f}%   p_e {100 * ours['p_e']:.2f}%   "
      f"kappa {ours['kappa']:.4f}   alpha {ours['alpha']:.4f}   (chapter's rule: 93.42%, 90.99%, 0.2698, 0.2655)")
print(f"scikit-learn's cohen_kappa_score on the same labels: {cohen_kappa_score(y_test, b_ours):.4f}")

A (reference, rows) vs our 22-phrase rule (columns), n = 10,000

                B: negative   B: positive
A: negative           200           432
A: positive           303         9,065

raw agreement 92.65%   p_e 89.29%   kappa 0.3140   alpha 0.3135   (chapter's rule: 93.42%, 90.99%, 0.2698, 0.2655)


scikit-learn's cohen_kappa_score on the same labels: 0.3140


### The same reference and rule, a different sample

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#the-same-annotators-a-different-number)

Take all 632 reviews A calls negative and a random 632 of those A calls positive. The
same two label sources, on a sample with a different class ratio. The chapter's table for
*its* rule:

```text
                    full test set (n=10,000)   balanced (n=1,264)
raw agreement                       93.42%              60.28%
p_e                                 90.99%              50.00%
Cohen's kappa                        0.2698              0.2057
Krippendorff's alpha                 0.2655              0.0722
```

The cell below does the same with our rule. The numbers differ from the chapter's; the
pattern is what to compare.

**What to look at:** raw agreement falls by tens of points, $p_e$ becomes exactly 0.5,
and kappa and alpha, which agreed on the full set, separate.

In [10]:
negatives = np.where(y_test == 0)[0]
positives = np.random.default_rng(42).choice(np.where(y_test == 1)[0], len(negatives), replace=False)
balanced = np.concatenate([negatives, positives])
bal = coefficients(table_of(y_test[balanced], b_ours[balanced]))

print(f"### our 22-phrase rule\n{'':<24}{'full test set (n=10,000)':>26}{'balanced (n=' + format(len(balanced), ',') + ')':>22}")
for label, key, fmt in [("raw agreement", "p_o", "{:.2%}"), ("p_e", "p_e", "{:.2%}"),
                        ("Cohen's kappa", "kappa", "{:.4f}"), ("Krippendorff's alpha", "alpha", "{:.4f}")]:
    print(f"{label:<24}{fmt.format(ours[key]):>26}{fmt.format(bal[key]):>22}")

### our 22-phrase rule
                          full test set (n=10,000)    balanced (n=1,264)
raw agreement                               92.65%                64.56%
p_e                                         89.29%                50.00%
Cohen's kappa                               0.3140                0.2911
Krippendorff's alpha                        0.3135                0.2053


**Reading it.** On the balanced subsample $p_e$ is exactly 0.5 whatever the rule does:
when one rater splits 50/50, the expected agreement is $0.5\,b + 0.5\,(1 - b) = 0.5$ for
any rate $b$ of the other. Alpha pools the two raters' labels and so penalises the rule
for calling far more than half of this sample positive; kappa does not. Neither column is
"the" agreement of these two sources. Each belongs to the class ratio of its sample,
which is why the chapter asks for the table and the marginals next to any coefficient.

### How wide is the interval around a kappa?

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#adjudication)

A pilot annotation has a few hundred items, not 10,000. The chapter resamples from its
table to see what a kappa of 0.27 looks like at that size: the same item's two judgements
are drawn together.

**What to look at:** the standard error at 200 randomly drawn items, and at 200 items
stratified as 100 reference-negative and 100 reference-positive.

In [11]:
rng = np.random.default_rng(42)
cell_probabilities = (BOOK_TABLE / BOOK_TABLE.sum()).ravel()


def kappa_draws(n_items, n_boot=2_000):
    """Kappa of n_boot samples of n_items (pairs of judgements) drawn from the chapter's table."""
    draws = rng.multinomial(n_items, cell_probabilities, size=n_boot).reshape(n_boot, 2, 2)
    return np.array([coefficients(t)["kappa"] for t in draws])


def kappa_se_stratified(n_per_class=100, n_boot=2_000):
    """The same, for a pilot of n reference-negative and n reference-positive items."""
    neg_hit = rng.binomial(n_per_class, BOOK_TABLE[0, 0] / BOOK_TABLE[0].sum(), n_boot)
    pos_hit = rng.binomial(n_per_class, BOOK_TABLE[1, 1] / BOOK_TABLE[1].sum(), n_boot)
    return np.nanstd([coefficients([[a, n_per_class - a], [n_per_class - d, d]])["kappa"]
                      for a, d in zip(neg_hit, pos_hit)])


draws_200 = kappa_draws(200)
vs_book("standard error, 10,000 items", np.nanstd(kappa_draws(10_000)), 0.02, "{:.2f}")
vs_book("standard error, 200 random items", np.nanstd(draws_200), 0.14, "{:.2f}")
vs_book("standard error, 100 + 100 stratified", kappa_se_stratified(), 0.04, "{:.2f}")
low, high = np.nanpercentile(draws_200, [2.5, 97.5])
print(f"\n200 random items: the middle 95% of the resampled kappas runs from {low:.2f} to {high:.2f}"
      "   (book: roughly [−0.02, 0.54])")

standard error, 10,000 items             this run       0.02   book       0.02   same as the book
standard error, 200 random items         this run       0.14   book       0.14   same as the book
standard error, 100 + 100 stratified     this run       0.04   book       0.04   same as the book

200 random items: the middle 95% of the resampled kappas runs from -0.03 to 0.53   (book: roughly [−0.02, 0.54])


The chapter's own bootstrap code works on two label vectors, which is what you will have
after a pilot. Here it is on our rule and the full test set.

In [12]:
a, b = y_test, b_ours
# the chapter's bootstrap
rng = np.random.default_rng(42)
idx = (rng.integers(0, len(a), len(a)) for _ in range(2000))
boot = [cohen_kappa_score(a[i], b[i]) for i in idx]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"our rule, n = {len(a):,}: kappa {cohen_kappa_score(a, b):.4f}, bootstrap 95% [{lo:.4f}, {hi:.4f}]")

our rule, n = 10,000: kappa 0.3140, bootstrap 95% [0.2768, 0.3503]


**Reading it.** At 200 randomly drawn items from a corpus this skewed, the interval
around 0.27 runs from about zero to about 0.54. A stratified pilot narrows it a great
deal, at a price: its kappa belongs to the 50/50 ratio you chose, not to the natural one.

## 6. Auditing labels at scale

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#sec-audit)

Nobody can read 40,000 rows. The chapter uses the model to decide which rows to read
first: train on four fifths of the training set, predict the remaining fifth, rotate, and
flag the rows where the model gives a high score to the class the label does *not* have.
It is a simple threshold rule inspired by confident learning, and the chapter says what
it is not: the score is not the probability that the label is wrong.

The code is the chapter's. Its comment matters: the vectoriser here was fitted on all
40,000 rows before the folds, so the predictions are not fully out-of-fold. The cell
after this one measures what changes when the vectoriser is fitted inside each fold.

**What to look at:** how few rows pass a high threshold, and how they split between the
two classes.

In [13]:
y = y_train
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Xtr here comes from a vectoriser fitted on all 40,000 rows, before the folds.
# For a fully out-of-fold prediction, fit the vectoriser inside each training fold too.
oof = np.zeros(len(y))
for train_idx, val_idx in skf.split(Xtr, y):
    oof[val_idx] = clf().fit(Xtr[train_idx], y[train_idx]).predict_proba(Xtr[val_idx])[:, 1]

# the score the model gives to the opposite class; not the probability that the label is wrong
wrong_conf = np.where(y == 1, 1 - oof, oof)

BOOK_FLAGS = {0.50: (3_026, 2_232, 794), 0.90: (92, 49, 43), 0.95: (34, 18, 16), 0.99: (2, 2, 0)}
for threshold, (b_total, b_pos, b_neg) in BOOK_FLAGS.items():
    flagged = wrong_conf > threshold
    print(f"> {threshold:.2f}")
    vs_book("  rows flagged", int(flagged.sum()), b_total, "{:,}")
    vs_book("  reference positive", int((flagged & (y == 1)).sum()), b_pos, "{:,}")
    vs_book("  reference negative", int((flagged & (y == 0)).sum()), b_neg, "{:,}")

flag = wrong_conf > 0.90
print()
vs_book("flag rate of negative labels, %", 100 * (flag & (y == 0)).sum() / (y == 0).sum(), 1.70, "{:.2f}")
vs_book("flag rate of positive labels, %", 100 * (flag & (y == 1)).sum() / (y == 1).sum(), 0.13, "{:.2f}")

> 0.50
  rows flagged                           this run      3,026   book      3,026   same as the book
  reference positive                     this run      2,232   book      2,232   same as the book
  reference negative                     this run        794   book        794   same as the book
> 0.90
  rows flagged                           this run         92   book         92   same as the book
  reference positive                     this run         49   book         49   same as the book
  reference negative                     this run         43   book         43   same as the book
> 0.95
  rows flagged                           this run         34   book         34   same as the book
  reference positive                     this run         18   book         18   same as the book
  reference negative                     this run         16   book         16   same as the book
> 0.99
  rows flagged                           this run          2   book          2   same as t

**Reading it.** The flags are close to class-balanced in a corpus that is 6.3% negative: a
negative label is flagged thirteen times as often as a positive one. That is a difference
in *flag* rates, not in error rates. The model may simply be less certain, or worse
calibrated, on the minority class.

Now the fully out-of-fold version: the vectoriser is fitted on the training part of each
fold only.

In [14]:
oof_infold = np.zeros(len(y))
for train_idx, val_idx in skf.split(Xtr, y):
    fold_vec = make_vec()
    X_fit = fold_vec.fit_transform(train["text"].iloc[train_idx])
    X_val = fold_vec.transform(train["text"].iloc[val_idx])
    oof_infold[val_idx] = clf().fit(X_fit, y[train_idx]).predict_proba(X_val)[:, 1]
wrong_conf_infold = np.where(y == 1, 1 - oof_infold, oof_infold)

vs_book("flags above 0.90, vectoriser in the fold", int((wrong_conf_infold > 0.90).sum()), 123)
vs_book("flags above 0.95, vectoriser in the fold", int((wrong_conf_infold > 0.95).sum()), 47)
vs_book("of the first 92, also in the new list", int((flag & (wrong_conf_infold > 0.90)).sum()), 89)

flags above 0.90, vectoriser in the fold this run        123   book        123   same as the book
flags above 0.95, vectoriser in the fold this run         47   book         47   same as the book
of the first 92, also in the new list    this run         89   book         89   same as the book


**Reading it.** Fitting the vectoriser outside the folds lets every fold see the
vocabulary and document frequencies of the rows it is about to score, and the model
looks a little more certain than it should. The list gets longer when that is fixed, and
almost all of the first list survives. In your own audit put the vectoriser and the
classifier in one `Pipeline`, which does this by itself. The numbers below belong to the
first list of 92, as in the chapter.

### A flag is a request to read

The 92 candidates are in the training set. In a separate audit the chapter's author read
the 40 *test* reviews on which the model disputes the stored label with the highest
score, and sorted them by hand: **11** `error` (the label clearly contradicts the text),
**20** `schema` (a single binary label cannot represent the text), **9** `model` (the
label is defensible and the classifier is wrong). Those are one unblinded reader's
judgements on a sample the model selected. The chapter says they are not an estimate for
the corpus or for the 92 training candidates, and that a second, blinded reader should
repeat the reading before the number is quoted.

The cell prints the same forty rows for you to read, checks the four the chapter
discusses, and computes the Wilson intervals the chapter gives for the two shares.

In [15]:
test_wrong_conf = np.where(y_test == 1, 1 - prob, prob)
top40 = np.argsort(-test_wrong_conf)[:40]

for row, book_label, book_p in [(6874, 0, 0.984), (8356, 1, 0.059), (8006, 1, 0.034), (2544, 1, None)]:
    print(f"row {row}: in the forty: {row in set(top40.tolist())}")
    vs_book("  reference label", int(y_test[row]), book_label)
    if book_p is not None:
        vs_book("  model p(positive)", prob[row], book_p, "{:.3f}")
print()
for label, k, book_low, book_high in [("schema share 20/40", 20, 35, 65), ("error share 11/40", 11, 16, 43)]:
    low, high = wilson(k, 40)
    vs_book(f"{label}: Wilson lower, %", 100 * low, book_low, "{:.0f}")
    vs_book(f"{label}: Wilson upper, %", 100 * high, book_high, "{:.0f}")

print("\nthe forty rows   (position, reference label, model p(positive), text)")
for row in top40:
    print(f"{row:>5}  {y_test[row]}  {prob[row]:.3f}  {test['text'].iloc[row][:140]}")

row 6874: in the forty: True
  reference label                        this run          0   book          0   same as the book
  model p(positive)                      this run      0.984   book      0.984   same as the book
row 8356: in the forty: True
  reference label                        this run          1   book          1   same as the book
  model p(positive)                      this run      0.059   book      0.059   same as the book
row 8006: in the forty: True
  reference label                        this run          1   book          1   same as the book
  model p(positive)                      this run      0.034   book      0.034   same as the book
row 2544: in the forty: True
  reference label                        this run          1   book          1   same as the book

schema share 20/40: Wilson lower, %      this run         35   book         35   same as the book
schema share 20/40: Wilson upper, %      this run         65   book         65   same as the book
e

For two of those rows the chapter looks at which features moved the model. In a linear
model each feature's contribution to the decision score is its TF-IDF value times its
coefficient.

**What to look at:** on row 8006 the word `iade` (return, refund) dominates. On row 2544
the bigram `fena değil` ("not bad") is in the vocabulary and contributes almost nothing;
the unigrams `değil` and `fena` decide.

In [16]:
feature_names = np.array(vec.get_feature_names_out())
coefficients_ = model.coef_[0]


def contributions(row):
    x = Xte[row]
    values = x.multiply(coefficients_).toarray()[0]
    return {feature_names[j]: values[j] for j in x.indices}


c = contributions(8006)
print(f"row 8006: {test['text'].iloc[8006][:110]}")
for feature, book_value in [("iade", -1.78), ("beğenmedim", -1.30), ("iade ettim", -0.65)]:
    vs_book(f"  contribution of '{feature}'", c[feature], book_value, "{:+.2f}")
negative_total = sum(v for v in c.values() if v < 0)
iade_share = sum(v for f, v in c.items() if "iade" in f and v < 0) / negative_total
print(f"  share of the negative contributions from features containing 'iade': {100 * iade_share:.0f}%   (book: about half)")

c = contributions(2544)
print(f"\nrow 2544: {test['text'].iloc[2544]}")
for feature, book_value in [("değil", -1.77), ("fena", -0.55), ("fena değil", -0.06)]:
    vs_book(f"  contribution of '{feature}'", c[feature], book_value, "{:+.2f}")

row 8006: hiç bir sebeb göstermeden iade kurallarına uyarak beğenmedim dedim, iade ettim.ücret iadem anında yapıldı.karg
  contribution of 'iade'                 this run      -1.78   book      -1.78   same as the book
  contribution of 'beğenmedim'           this run      -1.30   book      -1.30   same as the book
  contribution of 'iade ettim'           this run      -0.65   book      -0.65   same as the book
  share of the negative contributions from features containing 'iade': 51%   (book: about half)

row 2544: paketleme başarısız. ampüller fena değil
  contribution of 'değil'                this run      -1.77   book      -1.77   same as the book
  contribution of 'fena'                 this run      -0.55   book      -0.55   same as the book
  contribution of 'fena değil'           this run      -0.06   book      -0.06   same as the book


### What cleaning actually bought

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#what-cleaning-actually-bought)

Given 92 suspect rows, the obvious move is to drop them or flip their labels and retrain.
The chapter does both at three thresholds, and scores on the same test split. It calls
this an exploratory comparison, not a result on a never-seen test set.

**What to look at:** no row of the table is above the baseline.

In [17]:
def retrain(keep=None, labels=None):
    labels = y_train if labels is None else labels
    keep = np.ones(len(y_train), bool) if keep is None else keep
    return scores(y_test, clf().fit(Xtr[keep], labels[keep]).predict(Xte))


BOOK_REPAIR = {("drop", 0.90): (92.37, 75.72, 55.61), ("flip", 0.90): (92.47, 75.94, 55.99),
               ("drop", 0.95): (92.39, 75.83, 55.83), ("flip", 0.95): (92.52, 76.03, 56.15),
               ("drop", 0.99): (92.54, 76.15, 56.37), ("flip", 0.99): (92.52, 76.11, 56.31)}
repaired = {}
print("### repairs, judged on the test split used throughout the chapter")
print(f"{'':<24}{'acc':>7}{'macro-F1':>11}{'F1(neg)':>10}   book macro-F1")
print(f"{'baseline (no repair)':<24}{base_acc:>7.2f}{base_macro:>11.2f}{base_f1n:>10.2f}")
for threshold in (0.90, 0.95, 0.99):
    flagged = wrong_conf > threshold
    flipped = y_train.copy()
    flipped[flagged] = 1 - flipped[flagged]
    for action, result in [("drop", retrain(keep=~flagged)), ("flip", retrain(labels=flipped))]:
        repaired[(action, threshold)] = result
        book = BOOK_REPAIR[(action, threshold)]
        same = "same" if all(f"{r:.2f}" == f"{b:.2f}" for r, b in zip(result, book)) else "DIFFERS"
        print(f"{action} {int(flagged.sum()):>2}  (p > {threshold:.2f})      {result[0]:>7.2f}{result[1]:>11.2f}{result[2]:>10.2f}   {book[1]:.2f}  {same}")

### repairs, judged on the test split used throughout the chapter
                            acc   macro-F1   F1(neg)   book macro-F1
baseline (no repair)      92.57      76.21     56.47


drop 92  (p > 0.90)        92.37      75.72     55.61   75.72  same
flip 92  (p > 0.90)        92.47      75.94     55.99   75.94  same


drop 34  (p > 0.95)        92.39      75.83     55.83   75.83  same
flip 34  (p > 0.95)        92.52      76.03     56.15   76.03  same
drop  2  (p > 0.99)        92.54      76.15     56.37   76.15  same
flip  2  (p > 0.99)        92.52      76.11     56.31   76.11  same


A table like that needs a control. If dropping or flipping *any* 92 rows moves the score
about as much, the flagged rows are not special. The chapter draws 92 random rows with
the class mix of the flags (49 positive, 43 negative) twenty times and applies the same
two operations.

In [18]:
n_pos, n_neg = int((flag & (y_train == 1)).sum()), int((flag & (y_train == 0)).sum())
positive_rows, negative_rows = np.where(y_train == 1)[0], np.where(y_train == 0)[0]
rng = np.random.default_rng(0)
dropped, flipped_scores = [], []
for _ in range(20):
    pick = np.concatenate([rng.choice(positive_rows, n_pos, replace=False),
                           rng.choice(negative_rows, n_neg, replace=False)])
    keep = np.ones(len(y_train), bool)
    keep[pick] = False
    dropped.append(retrain(keep=keep)[1])
    labels = y_train.copy()
    labels[pick] = 1 - labels[pick]
    flipped_scores.append(retrain(labels=labels)[1])

print(f"### control: {n_pos + n_neg} random rows with the class mix of the flags ({n_pos} pos / {n_neg} neg), 20 draws")
for name, values, book in [("drop 92 random rows", dropped, (76.15, 0.13, 75.91, 76.45)),
                           ("flip 92 random rows", flipped_scores, (75.87, 0.36, 75.01, 76.47))]:
    vs_book(f"{name}: mean", np.mean(values), book[0], "{:.2f}")
    vs_book(f"{name}: sd", np.std(values, ddof=1), book[1], "{:.2f}")
    vs_book(f"{name}: min", min(values), book[2], "{:.2f}")
    vs_book(f"{name}: max", max(values), book[3], "{:.2f}")
print(f"\ndrop the 92 flagged rows   {repaired[('drop', 0.90)][1]:.2f}")
print(f"flip the 92 flagged rows   {repaired[('flip', 0.90)][1]:.2f}")

### control: 92 random rows with the class mix of the flags (49 pos / 43 neg), 20 draws
drop 92 random rows: mean                this run      76.15   book      76.15   same as the book
drop 92 random rows: sd                  this run       0.13   book       0.13   same as the book
drop 92 random rows: min                 this run      75.91   book      75.91   same as the book
drop 92 random rows: max                 this run      76.45   book      76.45   same as the book
flip 92 random rows: mean                this run      75.87   book      75.87   same as the book
flip 92 random rows: sd                  this run       0.36   book       0.36   same as the book
flip 92 random rows: min                 this run      75.01   book      75.01   same as the book
flip 92 random rows: max                 this run      76.47   book      76.47   same as the book

drop the 92 flagged rows   75.72
flip the 92 flagged rows   75.94


**Reading it.** Dropping the 92 flagged rows scores below the lowest of the twenty random
drops, and flipping them cannot be told apart from flipping 92 random labels. The flagged
rows do behave differently from random rows, but not by raising the score. Producing a
list worth reading and improving a model by editing that list wholesale are separate
goals. The second was not achieved here. Nobody read the 92 rows before they were dropped
or flipped; the chapter's advice is to read them first.

## 7. What label noise costs, and in which direction

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#what-label-noise-costs-and-in-which-direction)

Now the labels are corrupted on purpose. The training labels are flipped at a controlled
rate, the model is retrained, and it is scored against the original test labels. Three
regimes: `symmetric` flips the given share of *each* class, `neg -> pos` flips that share
of the negative labels only, `pos -> neg` that share of the positive labels only.

The rate is a share of the *source class*, and the two classes are very different in
size: 10% of the negatives is 253 labels, 10% of the positives is 3,747.

**Seeds.** The chapter reports the mean of three seeds and does not print which. This cell
uses seeds 0, 1 and 2 and prints the difference from the chapter's value. The chapter
reports that the standard deviation over its seeds never exceeded 0.50.

The cell trains the model 63 times and takes about ten seconds.

In [19]:
RATES = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.40]
NOISE_SEEDS = [0, 1, 2]
BOOK_NOISE = {"symmetric": [76.21, 74.18, 72.43, 68.58, 63.34, 55.43, 49.86, 43.71],
              "neg -> pos": [76.21, 76.14, 76.29, 76.27, 76.42, 76.85, 77.12, 76.47],
              "pos -> neg": [76.21, 74.17, 72.87, 68.63, 63.63, 56.69, 52.01, 47.99]}


def corrupt(labels, rate, regime, rng):
    """Flip `rate` of the source class(es): a share of the class, not of all rows."""
    noisy = labels.copy()
    sources = {"symmetric": (0, 1), "neg -> pos": (0,), "pos -> neg": (1,)}[regime]
    for source in sources:
        rows = np.where(labels == source)[0]
        chosen = rng.choice(rows, int(round(rate * len(rows))), replace=False)
        noisy[chosen] = 1 - noisy[chosen]
    return noisy


def predict_with(labels):
    return clf().fit(Xtr, labels).predict(Xte)


start = time.perf_counter()
noise_runs = {}
for regime in BOOK_NOISE:
    for rate in RATES:
        if rate == 0:
            noise_runs[(regime, rate)] = [pred]
        else:
            noise_runs[(regime, rate)] = [predict_with(corrupt(y_train, rate, regime, np.random.default_rng(seed)))
                                          for seed in NOISE_SEEDS]
print(f"### macro-F1, original test labels, model retrained at each rate; seeds {NOISE_SEEDS}   ({time.perf_counter() - start:.0f}s)")
print("flip rate   " + "".join(f"{rate:>7.0%}" for rate in RATES))
for regime in BOOK_NOISE:
    means = [np.mean([macro_f1(y_test, p) for p in noise_runs[(regime, rate)]]) for rate in RATES]
    print(f"{regime:<12}" + "".join(f"{m:>7.2f}" for m in means))
    print(f"{'  book':<12}" + "".join(f"{b:>7.2f}" for b in BOOK_NOISE[regime]))
    print(f"{'  Δ':<12}" + "".join(f"{m - b:>+7.2f}" for m, b in zip(means, BOOK_NOISE[regime])))
spread = max(np.std([macro_f1(y_test, p) for p in runs], ddof=1) for runs in noise_runs.values() if len(runs) > 1)
print(f"\nlargest standard deviation over seeds in this run: {spread:.2f}   (book: never above 0.50)")

### macro-F1, original test labels, model retrained at each rate; seeds [0, 1, 2]   (10s)
flip rate        0%     1%     2%     5%    10%    20%    30%    40%
symmetric     76.21  74.51  72.63  68.40  63.08  55.44  49.88  43.51
  book        76.21  74.18  72.43  68.58  63.34  55.43  49.86  43.71
  Δ           -0.00  +0.33  +0.20  -0.18  -0.26  +0.01  +0.02  -0.20
neg -> pos    76.21  76.23  76.17  76.33  76.58  76.67  77.11  77.11
  book        76.21  76.14  76.29  76.27  76.42  76.85  77.12  76.47
  Δ           -0.00  +0.09  -0.12  +0.06  +0.16  -0.18  -0.01  +0.64
pos -> neg    76.21  74.38  72.67  68.31  63.13  56.61  51.44  47.29
  book        76.21  74.17  72.87  68.63  63.63  56.69  52.01  47.99
  Δ           -0.00  +0.21  -0.20  -0.32  -0.50  -0.08  -0.57  -0.70

largest standard deviation over seeds in this run: 0.62   (book: never above 0.50)


**Reading it.** Two rows fall steeply and one barely moves. With this model and this class
ratio, flipping even 40% of the negative labels to positive leaves macro-F1 close to the
baseline, while flipping 10% of the positive labels to negative costs about twelve
points. The same *rate* is fifteen times as many rows in one direction as in the other.

The chapter looks at where the small rise in the middle row comes from: with 30% of the
negatives flipped, the model predicts "negative" less often, so precision on the negative
class rises and recall falls.

In [20]:
runs = noise_runs[("neg -> pos", 0.30)]
print(f"{'':<22}{'predicted negative':>20}{'precision (neg)':>18}{'recall (neg)':>15}")
print(f"{'baseline':<22}{int((pred == 0).sum()):>20,}{100 * precision_score(y_test, pred, pos_label=0):>17.1f}%"
      f"{100 * recall_score(y_test, pred, pos_label=0):>14.1f}%")
print(f"{'30% neg -> pos (mean)':<22}{np.mean([(p == 0).sum() for p in runs]):>20,.0f}"
      f"{100 * np.mean([precision_score(y_test, p, pos_label=0) for p in runs]):>17.1f}%"
      f"{100 * np.mean([recall_score(y_test, p, pos_label=0) for p in runs]):>14.1f}%")
print("book: about 845 instead of 1,075 predicted negative; precision 44.8% -> about 50%; recall 76.3% -> about 67%\n")
vs_book("baseline: predicted negative", int((pred == 0).sum()), 1_075, "{:,}")
vs_book("baseline: precision (neg), %", 100 * precision_score(y_test, pred, pos_label=0), 44.8, "{:.1f}")
vs_book("baseline: recall (neg), %", 100 * recall_score(y_test, pred, pos_label=0), 76.3, "{:.1f}")

                        predicted negative   precision (neg)   recall (neg)
baseline                             1,075             44.8%          76.3%
30% neg -> pos (mean)                  847             50.3%          67.4%
book: about 845 instead of 1,075 predicted negative; precision 44.8% -> about 50%; recall 76.3% -> about 67%

baseline: predicted negative             this run      1,075   book      1,075   same as the book
baseline: precision (neg), %             this run       44.8   book       44.8   same as the book
baseline: recall (neg), %                this run       76.3   book       76.3   same as the book


Is the asymmetry only a matter of how many rows change? The chapter flips the same
*number* of labels in each direction. Here the draws are fully specified
(`np.random.default_rng(seed).choice` over the rows of the source class, seeds 0, 1, 2),
so the cell compares at the book's precision.

In [21]:
COUNTS = [250, 500, 1000, 2000]
BOOK_EQUAL = {250: (76.35, 74.98), 500: (76.80, 73.64), 1000: (76.95, 71.11), 2000: (68.60, 67.91)}
print(f"### the same NUMBER of flipped training labels in each direction (seeds {NOISE_SEEDS}, mean macro-F1)")
for count in COUNTS:
    neg_to_pos, pos_to_neg = [], []
    for seed in NOISE_SEEDS:
        rng = np.random.default_rng(seed)
        for rows, out in [(negative_rows, neg_to_pos), (positive_rows, pos_to_neg)]:
            labels = y_train.copy()
            chosen = rng.choice(rows, count, replace=False)
            labels[chosen] = 1 - labels[chosen]
            out.append(macro_f1(y_test, predict_with(labels)))
    print(f"{count:>5} labels: {100 * count / len(negative_rows):.1f}% of the negatives, {100 * count / len(positive_rows):.2f}% of the positives")
    vs_book("   neg -> pos", np.mean(neg_to_pos), BOOK_EQUAL[count][0], "{:.2f}")
    vs_book("   pos -> neg", np.mean(pos_to_neg), BOOK_EQUAL[count][1], "{:.2f}")

### the same NUMBER of flipped training labels in each direction (seeds [0, 1, 2], mean macro-F1)


  250 labels: 9.9% of the negatives, 0.67% of the positives
   neg -> pos                            this run      76.35   book      76.35   same as the book
   pos -> neg                            this run      74.98   book      74.98   same as the book


  500 labels: 19.8% of the negatives, 1.33% of the positives
   neg -> pos                            this run      76.80   book      76.80   same as the book
   pos -> neg                            this run      73.64   book      73.64   same as the book


 1000 labels: 39.5% of the negatives, 2.67% of the positives
   neg -> pos                            this run      76.95   book      76.95   same as the book
   pos -> neg                            this run      71.11   book      71.11   same as the book


 2000 labels: 79.1% of the negatives, 5.34% of the positives
   neg -> pos                            this run      68.60   book      68.60   same as the book
   pos -> neg                            this run      67.91   book      67.91   same as the book


**Reading it.** With the same number of flipped labels the direction still matters, up to
1,000 labels: the difference is not explained by the row count alone. At 2,000 labels
four fifths of the negative class has been relabelled and both directions score low. The
result belongs to this model, this class ratio and this metric.

## 8. Your test labels change what the measurement means

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#your-test-set-is-a-ceiling-not-a-ruler)

Now the reverse. One fixed model, trained once on the original labels. Its predictions do
not change; the *test* labels do: a share of all test rows is flipped uniformly at random
and the same predictions are scored against the corrupted labels.

**Seeds.** The chapter reports the mean of five seeds and does not print which. This cell
uses seeds 0 to 4 and prints the difference from the chapter's value beside the
chapter's own standard deviation.

**What to look at:** the 2% row, next to the 73.51 that bag of words scored against the
original labels in chapter 4.

In [22]:
BOOK_TEST_NOISE = {0.01: (91.73, 74.68, 53.90, 0.23), 0.02: (90.85, 73.13, 51.31, 0.12), 0.05: (88.31, 69.41, 45.36, 0.17),
                   0.10: (84.19, 64.79, 38.66, 0.24), 0.20: (75.48, 57.00, 28.82, 0.36)}


def noisy_test_labels(rate, seed):
    rng = np.random.default_rng(seed)
    labels = y_test.copy()
    flipped = rng.random(len(labels)) < rate
    labels[flipped] = 1 - labels[flipped]
    return labels


print("### one fixed model, corrupted test labels, mean of 5 seeds (0-4)")
print(f"test noise      acc   macro-F1   F1(neg)   sd(macro-F1)     book macro-F1 (sd)      Δ")
print(f"    0%      {base_acc:7.2f} {base_macro:10.2f} {base_f1n:9.2f}           0.00")
for rate, book in BOOK_TEST_NOISE.items():
    rows = np.array([scores(noisy_test_labels(rate, seed), pred) for seed in range(5)])
    mean = rows.mean(axis=0)
    print(f"{rate:>6.0%}      {mean[0]:7.2f} {mean[1]:10.2f} {mean[2]:9.2f} {rows[:, 1].std(ddof=1):14.2f}"
          f"     {book[1]:>10.2f} ({book[3]:.2f})   {mean[1] - book[1]:+.2f}")

### one fixed model, corrupted test labels, mean of 5 seeds (0-4)
test noise      acc   macro-F1   F1(neg)   sd(macro-F1)     book macro-F1 (sd)      Δ
    0%        92.57      76.21     56.47           0.00
    1%        91.67      74.52     53.61           0.15          74.68 (0.23)   -0.16
    2%        90.80      73.11     51.29           0.08          73.13 (0.12)   -0.02
    5%        88.23      69.46     45.51           0.38          69.41 (0.17)   +0.05
   10%        83.84      64.37     38.04           0.34          64.79 (0.24)   -0.42
   20%        75.33      57.01     28.94           0.22          57.00 (0.36)   +0.01


**Reading it.** At 2% test-label noise the same model, with the same predictions, scores
about three points lower, below what a different model scored against the original
labels. The chapter is precise about what that comparison is: two *differently labelled*
test sets. It is not a reversal of rank on a shared one, and scores reported against
different versions of the labels cannot be compared directly.

Scoring both models against the *same* noisy labels is a different question, and the
chapter measured it: word bigram TF-IDF and bag of words, 200 draws of noise at each
rate. The draws are fully specified (seeds 0 to 199), so the cell compares at the book's
precision.

In [23]:
bow = CountVectorizer(min_df=2, token_pattern=PAT, **LOC)
pred_bow = clf().fit(bow.fit_transform(train["text"]), y_train).predict(bow.transform(test["text"]))
clean_gap = macro_f1(y_test, pred) - macro_f1(y_test, pred_bow)
vs_book("gap on the original labels", clean_gap, 2.69, "{:.2f}")

N_DRAWS = 200
for rate, book_gap in [(0.02, 2.50), (0.20, 1.36)]:
    gaps = []
    for seed in range(N_DRAWS):
        labels = noisy_test_labels(rate, seed)
        gaps.append(macro_f1(labels, pred) - macro_f1(labels, pred_bow))
    vs_book(f"mean gap at {rate:.0%} shared noise", np.mean(gaps), book_gap, "{:.2f}")
    print(f"   order reversed in {sum(g < 0 for g in gaps)} of {N_DRAWS} draws   (book: none of 200)")

gap on the original labels               this run       2.69   book       2.69   same as the book


mean gap at 2% shared noise              this run       2.50   book       2.50   same as the book
   order reversed in 0 of 200 draws   (book: none of 200)


mean gap at 20% shared noise             this run       1.36   book       1.36   same as the book
   order reversed in 0 of 200 draws   (book: none of 200)


**Reading it.** Under uniform, independent noise the gap shrinks and the order never
reverses. That is for flips scattered at random. Real label errors can cluster on the
hard rows where two models disagree, so the chapter also measures the worst case: how
few label changes, placed as badly as possible on the rows where two models disagree,
are enough to erase a gap. The search is greedy: change the one label that reduces the
gap most, and repeat until the gap is gone.

In [24]:
def changes_to_erase(pred_a, pred_b, limit=400):
    """Greedy worst case: label changes on disagreement rows until a's macro-F1 lead over b is gone."""
    labels = y_test.copy()
    disagree = np.where(pred_a != pred_b)[0]
    changes = 0
    while macro_f1(labels, pred_a) - macro_f1(labels, pred_b) > 0 and changes < limit:
        candidates = [i for i in disagree if pred_a[i] == labels[i]]       # a is right there, b is wrong
        candidates = candidates[:60] + candidates[-60:]
        best_gap, best_row = None, None
        for i in candidates:
            labels[i] = 1 - labels[i]
            gap = macro_f1(labels, pred_a) - macro_f1(labels, pred_b)
            labels[i] = 1 - labels[i]
            if best_gap is None or gap < best_gap:
                best_gap, best_row = gap, i
        labels[best_row] = 1 - labels[best_row]
        changes += 1
    return changes, len(disagree)


# chapter 4's 1.2-point pair: unigram tf-idf with C = 10 against bag of words
uni = TfidfVectorizer(min_df=2, sublinear_tf=True, token_pattern=PAT, **LOC)
pred_uni = clf(10.0).fit(uni.fit_transform(train["text"]), y_train).predict(uni.transform(test["text"]))

changes, rows = changes_to_erase(pred_uni, pred_bow)
vs_book("1.2-point gap: rows the two disagree on", rows, 231)
vs_book("1.2-point gap: label changes to erase it", changes, 18)
changes, rows = changes_to_erase(pred, pred_bow)
vs_book("2.69-point gap: label changes to erase it", changes, 41)

effects = []
for i in list(np.where(y_test == 0)[0][:150]) + list(np.where(y_test == 1)[0][:150]):
    labels = y_test.copy()
    labels[i] = 1 - labels[i]
    effects.append(abs(macro_f1(labels, pred) - base_macro))
print(f"\none changed label moves one model's macro-F1 by {min(effects):.3f} to {max(effects):.3f} points   (book: 0.02 to 0.045)")

1.2-point gap: rows the two disagree on  this run        231   book        231   same as the book
1.2-point gap: label changes to erase it this run         18   book         18   same as the book


2.69-point gap: label changes to erase it this run         41   book         41   same as the book

one changed label moves one model's macro-F1 by 0.019 to 0.045 points   (book: 0.02 to 0.045)


**Reading it.** Eighteen labels out of 10,000, in the worst places, erase a 1.2-point
gap. With randomly scattered errors the gap survives, as the previous cell showed. So a
small gap between two models is not a result until the rows they disagree on have been
read, blind if possible, and the gap has been tested on one common version of the labels
(chapter 4's paired bootstrap).

## 9. Leakage: the errors that make you look good

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#sec-leakage)

Label noise usually lowers a score. Leakage raises it. The commonest form is the same
text on both sides of the split, so the chapter first counts the duplicates in the raw
file, and then makes the ordinary mistake on purpose: split first, de-duplicate never.

**What to look at:** 1,657 extra rows is not 1,657 duplicate groups, and 77 of the groups
carry both labels.

In [25]:
group_sizes = raw["text"].value_counts()
duplicate_groups = group_sizes[group_sizes > 1]
labels_per_text = raw.groupby("text")["label"].nunique()
vs_book("rows in the raw file", len(raw), 235_165, "{:,}")
vs_book("extra rows removed by de-duplication", len(raw) - raw["text"].nunique(), 1_657, "{:,}")
vs_book("  share of the file, %", 100 * (len(raw) - raw["text"].nunique()) / len(raw), 0.70, "{:.2f}")
vs_book("texts that occur more than once", len(duplicate_groups), 1_055, "{:,}")
vs_book("rows in those groups", int(duplicate_groups.sum()), 2_712, "{:,}")
vs_book("groups that carry both labels", int((labels_per_text > 1).sum()), 77)

# the careless version: sample and split the raw file, without de-duplicating first
leaky_sub, _ = train_test_split(raw, train_size=50_000, stratify=raw["label"], random_state=42)
leaky_train, leaky_test = train_test_split(leaky_sub, test_size=0.2, stratify=leaky_sub["label"], random_state=42)
leaked = leaky_test["text"].isin(set(leaky_train["text"])).to_numpy()
leaky_vec = make_vec()
leaky_pred = clf().fit(leaky_vec.fit_transform(leaky_train["text"]), leaky_train["label"]).predict(leaky_vec.transform(leaky_test["text"]))
y_leaky = leaky_test["label"].to_numpy()

print("\n### split-before-dedup, the ordinary careless mistake")
vs_book("test rows also present in train", int(leaked.sum()), 45)
vs_book("accuracy on the leaked rows", 100 * accuracy_score(y_leaky[leaked], leaky_pred[leaked]), 93.33, "{:.2f}")
vs_book("accuracy on the other rows", 100 * accuracy_score(y_leaky[~leaked], leaky_pred[~leaked]), 91.90, "{:.2f}")
vs_book("macro-F1 on the leaked rows", macro_f1(y_leaky[leaked], leaky_pred[leaked]), 76.76, "{:.2f}")
vs_book("macro-F1 on the other rows", macro_f1(y_leaky[~leaked], leaky_pred[~leaked]), 74.08, "{:.2f}")
train_label = leaky_train.drop_duplicates("text").set_index("text")["label"]
conflicts = sum(int(train_label[t]) != int(l) for t, l in zip(leaky_test["text"][leaked], y_leaky[leaked]))
vs_book("leaked rows that are positive", int(y_leaky[leaked].sum()), 42)
vs_book("median tokens of a leaked row", int(np.median([len(tokenize(t)) for t in leaky_test["text"][leaked]])), 2)
vs_book("leaked rows whose train copy has the other label", conflicts, 5)

rows in the raw file                     this run    235,165   book    235,165   same as the book
extra rows removed by de-duplication     this run      1,657   book      1,657   same as the book
  share of the file, %                   this run       0.70   book       0.70   same as the book
texts that occur more than once          this run      1,055   book      1,055   same as the book
rows in those groups                     this run      2,712   book      2,712   same as the book
groups that carry both labels            this run         77   book         77   same as the book



### split-before-dedup, the ordinary careless mistake
test rows also present in train          this run         45   book         45   same as the book
accuracy on the leaked rows              this run      93.33   book      93.33   same as the book
accuracy on the other rows               this run      91.90   book      91.90   same as the book
macro-F1 on the leaked rows              this run      76.76   book      76.76   same as the book
macro-F1 on the other rows               this run      74.08   book      74.08   same as the book
leaked rows that are positive            this run         42   book         42   same as the book
median tokens of a leaked row            this run          2   book          2   same as the book
leaked rows whose train copy has the other label this run          5   book          5   same as the book


**Reading it.** Forty-five leaked rows out of 10,000 cannot move the headline number, and
the 2.68-point difference between the two subsets is not the gain of memorisation: the
leaked rows are almost all positive and two tokens long. In five of them the copy in
training carries the *other* label, so a duplicate is also a place to look for label
problems. The 74.08 here is not comparable with the 76.21 baseline either, because this
is a different split.

Near-duplicates next: the properly de-duplicated split, with case, punctuation and
whitespace normalised away before comparing. The chapter's table says 34 test rows; its
own recount, noted in the text, gives 32 "depending on the details of the normalisation".
This cell uses the recount's normalisation.

In [26]:
def normalise(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", tr_lower(s))).strip()


train_normalised = {normalise(s) for s in train["text"]}
near = np.array([(normalise(s) in train_normalised) and normalise(s) != "" for s in test["text"]])
vs_book("near-duplicate test rows", int(near.sum()), 34, note="the chapter's recount gives 32")
vs_book("  of which positive (recount)", int(y_test[near].sum()), 31)
vs_book("  median tokens (recount)", int(np.median(n_tokens[near])), 3)
print("  examples:", ", ".join(repr(s) for s in test["text"][near].head(6)))

near-duplicate test rows                 this run         32   book         34   not comparable: the chapter's recount gives 32
  of which positive (recount)            this run         31   book         31   same as the book
  median tokens (recount)                this run          3   book          3   same as the book
  examples: 'hizli kargo', 'ürün  güzel tavsiye ederim', 'mükemmel***********', 'iyi iş', 'tavsi̇ye ederi̇m,', 'teşekkürler hb.'


To see how the effect grows with the dose, the chapter copies a controlled share of the
test rows into the training data and refits the vectoriser on the enlarged set. The 0%
and 100% rows have no random draw. For the other doses the chapter does not print its
draw; it reports that three draws of the 10% dose ranged from 76.81 to 77.13.

**What to look at:** the 10% row. It is the size of a genuine modelling improvement.

In [27]:
BOOK_DOSE = {0.0: (92.57, 76.21, 56.47), 0.01: (92.57, 76.27, 56.60), 0.02: (92.62, 76.41, 56.86), 0.05: (92.65, 76.53, 57.08),
             0.10: (92.82, 77.14, 58.20), 0.25: (93.33, 78.59, 60.82), 0.50: (94.00, 80.89, 65.07), 1.0: (95.62, 85.89, 74.17)}


def contaminated(share, seed):
    rng = np.random.default_rng(seed)
    copied = rng.choice(len(y_test), int(round(share * len(y_test))), replace=False)
    texts = pd.concat([train["text"], test["text"].iloc[copied]])
    labels = np.concatenate([y_train, y_test[copied]])
    v = make_vec()
    return scores(y_test, clf().fit(v.fit_transform(texts), labels).predict(v.transform(test["text"])))


print("test rows copied into train    acc   macro-F1   F1(neg)     book macro-F1")
for share in BOOK_DOSE:
    if share in (0.0, 1.0):
        result = contaminated(share, 42)
        verdict = "same as the book" if f"{result[1]:.2f}" == f"{BOOK_DOSE[share][1]:.2f}" else "DIFFERS from the book"
        print(f"{share:>26.0%} {result[0]:8.2f} {result[1]:10.2f} {result[2]:9.2f} {BOOK_DOSE[share][1]:>13.2f}   {verdict}")
    else:
        results = np.array([contaminated(share, seed) for seed in (0, 1, 2)])
        mean = results.mean(axis=0)
        print(f"{share:>26.0%} {mean[0]:8.2f} {mean[1]:10.2f} {mean[2]:9.2f} {BOOK_DOSE[share][1]:>13.2f}   "
              f"{mean[1] - BOOK_DOSE[share][1]:+.2f}, three draws {results[:, 1].min():.2f}–{results[:, 1].max():.2f}")

test rows copied into train    acc   macro-F1   F1(neg)     book macro-F1


                        0%    92.57      76.21     56.47         76.21   same as the book


                        1%    92.58      76.29     56.65         76.27   +0.02, three draws 76.28–76.31


                        2%    92.57      76.26     56.59         76.41   -0.15, three draws 76.14–76.34


                        5%    92.71      76.65     57.28         76.53   +0.12, three draws 76.52–76.81


                       10%    92.86      77.13     58.17         77.14   -0.01, three draws 76.95–77.31


                       25%    93.35      78.78     61.20         78.59   +0.19, three draws 78.65–79.01


                       50%    94.10      81.01     65.24         80.89   +0.12, three draws 80.83–81.12


                      100%    95.62      85.89     74.17         85.89   same as the book


**Reading it.** Copying every test row into training takes macro-F1 from 76.21 to 85.89,
and a 10% contamination gives about one point: a difference that looks like a modelling
improvement and cannot be told apart from one by looking at the score. The effect belongs
to this model, this dataset and this way of copying. It is not a fixed gain per 10%. The
chapter lists six kinds of leak to check separately; duplicates are only the first.

## 10. Where to spend the next thousand labels

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#where-to-spend-the-next-thousand-labels)

A fixed budget buys many cheap labels or fewer careful ones. The chapter's grid crosses
the number of labelled reviews with a symmetric noise rate.

**The protocol differs from chapter 4's learning curve.** Here the rows are drawn at
random and the vectoriser is refitted on *those rows only*: someone with 1,000 labelled
reviews and no other text. In chapter 4 the vocabulary and IDF came from all 40,000
texts and only the classifier saw fewer labels. The two give very different numbers at
small sizes, so do not carry a number from one table into the other.

**Seeds.** The chapter reports the mean of three seeds and does not print which. This cell
uses seeds 0, 1 and 2 and prints the differences. Small cells move a lot from seed to
seed: over ten seeds the chapter measured a standard deviation of 2.24 points for the
1,000-label clean cell.

The cell is 126 trainings and takes about half a minute.

In [28]:
BOOK_GRID = {500: [59.44, 58.56, 55.57, 51.18, 47.40, 42.12], 1_000: [63.71, 60.91, 57.77, 52.32, 46.99, 42.78],
             2_500: [68.76, 64.17, 60.03, 53.36, 48.23, 42.93], 5_000: [71.69, 66.00, 60.21, 53.17, 47.86, 42.42],
             10_000: [73.72, 66.47, 61.81, 54.46, 48.53, 42.88], 20_000: [74.89, 66.99, 62.04, 55.21, 49.09, 43.81],
             40_000: [76.21, 68.18, 63.20, 55.74, 49.34, 44.16]}
GRID_RATES = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40]
GRID_SEEDS = [0, 1, 2]


def budget_cell(n_labels, rate, seed):
    """Random n rows, vectoriser refitted on those rows, symmetric label noise, original test labels."""
    rng = np.random.default_rng(seed)
    if n_labels == len(y_train):
        X, X_eval, labels = Xtr, Xte, y_train.copy()
    else:
        rows = rng.choice(len(y_train), n_labels, replace=False)
        v = make_vec()
        X, X_eval, labels = v.fit_transform(train["text"].iloc[rows]), v.transform(test["text"]), y_train[rows].copy()
    if rate:
        flipped = rng.random(len(labels)) < rate
        labels[flipped] = 1 - labels[flipped]
    return macro_f1(y_test, clf().fit(X, labels).predict(X_eval))


start = time.perf_counter()
grid = {(n_labels, rate): np.mean([budget_cell(n_labels, rate, seed) for seed in GRID_SEEDS])
        for n_labels in BOOK_GRID for rate in GRID_RATES}
print(f"### macro-F1 by budget and label noise (original test labels, seeds {GRID_SEEDS})   ({time.perf_counter() - start:.0f}s)")
print(f"{'':<10}" + "".join(f"{rate:>8.0%}" for rate in GRID_RATES))
for n_labels in BOOK_GRID:
    print(f"{n_labels:>8,}  " + "".join(f"{grid[(n_labels, rate)]:>8.2f}" for rate in GRID_RATES))
    print(f"{'  Δ book':<10}" + "".join(f"{grid[(n_labels, rate)] - b:>+8.2f}" for rate, b in zip(GRID_RATES, BOOK_GRID[n_labels])))

### macro-F1 by budget and label noise (original test labels, seeds [0, 1, 2])   (37s)
                0%      5%     10%     20%     30%     40%
     500     58.25   57.66   56.57   50.96   46.01   42.07
  Δ book     -1.19   -0.90   +1.00   -0.22   -1.39   -0.05
   1,000     63.16   60.69   57.77   52.75   47.98   42.90
  Δ book     -0.55   -0.22   -0.00   +0.43   +0.99   +0.12
   2,500     69.22   63.56   59.26   51.30   46.59   41.60
  Δ book     +0.46   -0.61   -0.77   -2.06   -1.64   -1.33
   5,000     72.30   66.18   61.09   53.55   47.95   43.41
  Δ book     +0.61   +0.18   +0.88   +0.38   +0.09   +0.99
  10,000     73.76   66.77   61.45   54.80   49.00   43.85
  Δ book     +0.04   +0.30   -0.36   +0.34   +0.47   +0.97
  20,000     75.10   68.34   62.91   55.15   49.14   43.85
  Δ book     +0.21   +1.35   +0.87   -0.06   +0.05   +0.04
  40,000     76.21   68.58   62.60   55.41   49.38   43.99
  Δ book     -0.00   +0.40   -0.60   -0.33   +0.04   -0.17


**Reading it.** Read down a column and more labels help. Read along a row and noise costs
more than volume buys: the clean column at 2,500 labels is close to the 5%-noise column
at 40,000. The differences from the chapter's cells are largest in the small clean cells,
which is where three-seed means are least stable.

The chapter's text compares two pairs of cells and re-ran them with **ten** seeds each,
because a three-seed difference of half a point settles nothing. Seeds 0 to 9 are
specified, so this comparison is at the book's precision.

In [29]:
BOOK_TEN_SEEDS = {(500, 0.0): 59.20, (1_000, 0.0): 62.24, (2_500, 0.0): 69.26,
                  (40_000, 0.05): 68.38, (40_000, 0.10): 62.66, (40_000, 0.20): 55.32}
ten = {cell: np.array([budget_cell(cell[0], cell[1], seed) for seed in range(10)]) for cell in BOOK_TEN_SEEDS}
for cell, book in BOOK_TEN_SEEDS.items():
    vs_book(f"{cell[0]:>6,} labels, {cell[1]:.0%} noise: mean", ten[cell].mean(), book, "{:.2f}")
    print(f"      sd {ten[cell].std(ddof=1):.2f}, range {ten[cell].min():.2f}–{ten[cell].max():.2f}")
print()
for a, b, book_ahead in [((1_000, 0.0), (40_000, 0.10), 37), ((500, 0.0), (40_000, 0.20), 100)]:
    ahead = int((ten[a][:, None] > ten[b][None, :]).sum())
    vs_book(f"{a[0]:,} clean ahead of 40,000 at {b[1]:.0%}, of 100 pairs", ahead, book_ahead)

   500 labels, 0% noise: mean            this run      59.20   book      59.20   same as the book
      sd 1.19, range 57.57–61.13
 1,000 labels, 0% noise: mean            this run      62.24   book      62.24   same as the book
      sd 2.24, range 59.29–65.44
 2,500 labels, 0% noise: mean            this run      69.26   book      69.26   same as the book
      sd 1.27, range 67.02–71.14
40,000 labels, 5% noise: mean            this run      68.38   book      68.38   same as the book
      sd 0.48, range 67.61–69.28
40,000 labels, 10% noise: mean           this run      62.66   book      62.66   same as the book
      sd 0.46, range 61.92–63.27
40,000 labels, 20% noise: mean           this run      55.32   book      55.32   same as the book
      sd 0.47, range 54.68–56.14

1,000 clean ahead of 40,000 at 10%, of 100 pairs this run         37   book         37   same as the book
500 clean ahead of 40,000 at 20%, of 100 pairs this run        100   book        100   same as the book


**Reading it.** Over ten seeds, 1,000 clean labels and 40,000 labels at 10% noise cannot
be told apart: the clean cell is ahead in about a third of the seed pairs, and its own
seed-to-seed spread is larger than the difference. 500 clean labels against 40,000 at 20%
noise is a different matter; the clean cell is ahead in every pair.

The break-even rate at which 5,000 clean labels equal 40,000 noisy ones is not in the
grid. The chapter reads it off a straight line between the two measured cells on either
side and labels it as an interpolation.

In [30]:
clean_5000, at_0, at_5 = BOOK_GRID[5_000][0], BOOK_GRID[40_000][0], BOOK_GRID[40_000][1]
break_even = 0.05 * (at_0 - clean_5000) / (at_0 - at_5)
print(f"5,000 clean labels: {clean_5000}; 40,000 labels: {at_0} at 0% noise, {at_5} at 5%")
vs_book("break-even noise rate, interpolated, %", 100 * break_even, 2.8, "{:.1f}")
print("an interpolation between two measured cells of the chapter's grid, not a measured threshold")

5,000 clean labels: 71.69; 40,000 labels: 76.21 at 0% noise, 68.18 at 5%
break-even noise rate, interpolated, %   this run        2.8   book        2.8   same as the book
an interpolation between two measured cells of the chapter's grid, not a measured threshold


### Not all labels are worth the same

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#not-all-labels-are-worth-the-same)

If you can choose which rows to label, choose the ones the current model is least sure
about. The chapter's schematic: train on 200 random labels, score the unlabelled pool,
and take the 800 examples scored closest to 0.5.

The chapter prints one table (random 1,000: 65.49 macro-F1 with 62 negatives; 200 + 800
uncertain: 68.80 with 248 negatives) and then says its own re-run, with a protocol it
spells out, gave gains of 5.3 and 5.2 points in two groups of three seeds. This cell
implements that spelled-out protocol: the seed model reads the vectoriser of the 40,000
texts, and the final model's vectoriser is fitted on the 1,000 selected rows. Seeds 0, 1
and 2 are one of the chapter's two groups.

**What to look at:** how many negative reviews each strategy finds.

In [31]:
def spend_two_ways(seed):
    order = np.random.default_rng(seed).permutation(len(y_train))
    random_rows, seed_rows, pool = order[:1_000], order[:200], order[200:]
    # schematic of the chapter: 200 random labels, then the 800 pool examples scored closest to 0.5
    m0 = clf().fit(Xtr[seed_rows], y_train[seed_rows])
    unc = np.abs(m0.predict_proba(Xtr[pool])[:, 1] - 0.5)
    picks = pool[np.argsort(unc)[:800]]
    out = {}
    for name, rows in [("random", random_rows), ("uncertain", np.concatenate([seed_rows, picks]))]:
        v = make_vec()
        p = clf().fit(v.fit_transform(train["text"].iloc[rows]), y_train[rows]).predict(v.transform(test["text"]))
        out[name] = (int((y_train[rows] == 0).sum()),) + scores(y_test, p)
    return out


AL_SEEDS = [0, 1, 2]
runs = [spend_two_ways(seed) for seed in AL_SEEDS]
print(f"### 1,000 labels, spent two ways (seeds {AL_SEEDS})")
print(f"{'':<30}negatives found    acc   macro-F1   F1(neg)")
means = {}
for name, label in [("random", "1,000 drawn at random"), ("uncertain", "200 random + 800 uncertain")]:
    means[name] = np.mean([run[name] for run in runs], axis=0)
    print(f"{label:<30}{means[name][0]:>15.0f} {means[name][1]:>6.2f} {means[name][2]:>10.2f} {means[name][3]:>9.2f}")
print()
vs_book("random 1,000: macro-F1", means["random"][2], 63.13, "{:.2f}")
vs_book("200 + 800 uncertain: macro-F1", means["uncertain"][2], 68.46, "{:.2f}")
vs_book("gain, points", means["uncertain"][2] - means["random"][2], 5.3, "{:.1f}")
vs_book("negatives found, uncertain", means["uncertain"][0], 290, "{:.0f}")

### 1,000 labels, spent two ways (seeds [0, 1, 2])
                              negatives found    acc   macro-F1   F1(neg)
1,000 drawn at random                      60  93.16      63.13     29.85
200 random + 800 uncertain                290  93.75      68.46     40.22

random 1,000: macro-F1                   this run      63.13   book      63.13   same as the book
200 + 800 uncertain: macro-F1            this run      68.46   book      68.46   same as the book
gain, points                             this run        5.3   book        5.3   same as the book
negatives found, uncertain               this run        290   book        290   same as the book


**Reading it.** Uncertainty sampling finds several times as many negative reviews and
scores higher with the same 1,000 labels. The direction is the same as in the chapter's
printed table; the size depends on the protocol and on the seed, which is why the chapter
gives both. Three limits come with it. A set chosen by uncertainty does not represent the
natural distribution, so never build a *test* set this way. The selection also changes
how hard the texts are, so the gain is not explained by the number of negatives alone.
And the set is tied to the model that selected it.

## 11. Exercises

These follow [chapter 5's own exercises](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html#exercises), using the code above. The numbers in brackets are the
chapter's exercise numbers.

1. **Two tables for one raw agreement** [2]. With `coefficients` from section 5, build the
   2×2 table for two annotators who agree on 90 of 100 items when both use the positive
   label 50 times, and when both use it 95 times. Show that the marginals and the 90
   agreements leave only one possible table in each case. Then keep raw agreement at 90%,
   change the marginals, and watch kappa move.
2. **Your own audit** [3]. Change the seed in section 3 and read a different hundred rows.
   Write your criteria for *error*, *undecidable* and *ok* before you start. If you find
   no outright error, report the rule of three; if you find some, report `wilson(k, 100)`.
3. **A balanced corpus** [4]. Build a class-balanced training set from `train` and rerun
   the two cells of section 7. Does the asymmetry between the two directions survive?
   Explain the result from the class counts.
4. **A second annotator of your own** [5]. Replace `NEG` and `POS` in section 5 with a
   rule you write for your own task, or with a regular expression. Report kappa on the
   full test set and on the balanced subsample, with the 2×2 table and both raters' rates.
5. **Out-of-fold, properly** [6]. Use the in-fold scores of section 6
   (`wrong_conf_infold`). Read the twenty highest-scoring candidates without looking at
   the stored label. Then compare three conditions on the test split: drop them, flip
   them, repair only the ones you verified by hand. Run the random-row control for each.
6. **The break-even, for another budget** [8]. From `BOOK_GRID` in section 10, find between
   which two measured noise levels 2,500 clean labels equal 40,000 noisy ones, compute the
   interpolated rate and label it as an interpolation. Then say what the table does not
   contain for deciding how to split a budget between training and test labels.
7. **Read the conflicting duplicates** [9]. Section 9 counted 77 duplicate groups that
   carry both labels. List them with
   `raw[raw["text"].isin(labels_per_text[labels_per_text > 1].index)]`, read them, and
   separate clear error, missing context and schema ambiguity. Say in one sentence why
   their conflict rate is not comparable with the random-100 audit.
8. **Uncertainty sampling for another model** [the chapter's third limit]. In
   `spend_two_ways`, keep the rows chosen by the TF-IDF seed model but train the final
   model on a different representation, for example character n-grams from chapter 4.
   Does the selected set still beat the random one?

The lab at the end of the chapter needs no code: two people, two balanced rounds of 100
reviews, a guideline written between the rounds, and the agreement measures of section 5
on your own labels.

---

*Code in this notebook is MIT-licensed; the prose is CC BY 4.0. The Turkish product
reviews dataset is CC BY-SA 4.0 (Fatih Barmanbay, `fthbrmnby/turkish_product_reviews` on
Hugging Face); the review texts printed in this notebook's outputs come from it.*